# CH4 Economics workspace
Execute the verified September 29 Drive package. Economics is available only for 2019 and experimental 2021. Native asset ingestion must finish before verification. This notebook preserves yield equations and grid.

In [ ]:
from pathlib import Path
from google.colab import drive
import ee,sys,importlib,json,unittest
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/PHD/CSP3_GPP_outputs')
try:
    ee.Initialize(project='ee-njberkowitz95')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='ee-njberkowitz95')


In [ ]:
# CH4 Economics workspace: latest verified inputs
from pathlib import Path
import sys, importlib, unittest, json
ROOT=Path('/content/drive/MyDrive/PHD/CSP3_GPP_outputs')
PROFIT_CODE=ROOT/'CH4_marginality/20260929_profit_app_v1/code'
assert (PROFIT_CODE/'ch4_marginality/profit_workspace.py').exists()
sys.path.insert(0,str(PROFIT_CODE))
for name in list(sys.modules):
    if name=='ch4_marginality' or name.startswith('ch4_marginality.') or name.startswith('test_'):
        del sys.modules[name]
importlib.invalidate_caches()
suite=unittest.TestLoader().discover(str(PROFIT_CODE/'ch4_marginality/tests'))
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()
from ch4_marginality.profit_workspace import run as run_profit
PROFIT_OUTPUT=run_profit(ROOT)
from ch4_marginality.profit_assets import submit
profit_tasks=submit(ROOT,'testernoah135')
print('Colab profit evidence completed; asset submissions:',len(profit_tasks))


In [ ]:
# Native annual patch/index verification
import ch4_marginality.profit_assets as profit_assets
import ee,json,importlib
importlib.reload(profit_assets)
tasks=json.loads((PROFIT_OUTPUT/'asset_tasks.json').read_text())
states=ee.data.getTaskStatus([t['response']['id'] for t in tasks if 'response' in t])
print([(s['state'],s.get('error_message','')) for s in states])
assert all(s['state']=='COMPLETED' for s in states), 'Asset ingestion pending or failed.'
checks=profit_assets.verify(ROOT)
print('Verified annual patch geometries and native identities:',len(checks))

In [ ]:
# Native profit layers, missing yield, and source-scope edges
from ch4_marginality.verify_profit_pixels import run as verify_profit_pixels
pixel_checks=verify_profit_pixels(ROOT)
print('Native economic source/model checks:',len(pixel_checks))

In [ ]:
# Final platform provenance, freshness gate, and verified candidate
import importlib,sys
for name in list(sys.modules):
    if name=='ch4_marginality' or name.startswith('ch4_marginality.'):
        del sys.modules[name]
importlib.invalidate_caches()
from ch4_marginality.finalize_profit import run as finalize_profit
CANDIDATE=finalize_profit(ROOT)
from ch4_marginality.profit_report import run as export_profit_report
export_profit_report(ROOT)
print('Academic exports and verified candidate completed:',PROFIT_OUTPUT)